# Session 2 — Iteration 7: Class-Based RAG Architecture

Refactors the ingestion pipeline into a clean `VectorDB` class inspired by the
[Anthropic Contextual Embeddings recipe](https://platform.claude.com/cookbook/capabilities-contextual-embeddings-guide).

**Architecture pattern:**
- `VectorDB` — baseline: chunk → embed → store in ChromaDB
- `ContextualVectorDB` — extends baseline: Claude situates each chunk within its source PDF, then embeds `context + chunk`.  This contextualization is used to improve retrieval rates by adding additional "context" to embeddings. This improved, embedded chunk is what is added to our prompt context before sending to the LLM for answer. 
- Prompt caching — full PDF text is cached once per document; subsequent chunks read from cache (~90% discount, much savings)

**Key details:**
- ChromaDB `PersistentClient` — index survives kernel restarts
- Contextualization runs **sequentially per PDF** to maximize cache hits
- Search returns **original chunk text** for citations; embeddings use contextualized text


In [ ]:
import csv
import os
import re
import tiktoken
import chromadb
import openai as openai_client
import anthropic
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError
from dotenv import load_dotenv, find_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv(find_dotenv())

# Model names — change here to update everywhere.
# Anthropic does not haave their own embedding model.  Anthropic recommends Voyage AI, but I do not have an account with them. 
# So, we will use OpenAI's embedding model because I already have an account with them.  
# We will use Anthropic's LLM however, as I have an account with them as well and prefer that over OpenAI
EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL           = "claude-sonnet-4-5"
CONTEXTUALIZE_MODEL = os.environ.get("RAG_CONTEXTUALIZE_MODEL", "claude-haiku-4-5")
MAX_DOC_TOKENS      = int(os.environ.get("RAG_MAX_DOC_TOKENS", 80_000))  # truncate very large PDFs for Claude

/var/folders/1f/srb1v76j181_9p65kq4xmmjw0000gn/T/ipykernel_56702/1155121712.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
# Regex to find section headings inside a text chunk (used for citation metadata)
_HEADING_RE = re.compile(
    r'(?:Chapter\s+[A-Z]?\d+[^\n]*|ARTICLE\s+[IVXLC]+[^\n]*|§\s*[A-Z]?\d+[-\w]*\.[^\n]*)',
    re.MULTILINE,
)

def _extract_heading(text: str, page: int) -> str:
    """Return the first section header found in the chunk, or fall back to page number."""
    match = _HEADING_RE.search(text)
    if match:
        return match.group(0).strip()[:80]
    return f"p. {page + 1}"

In [ ]:
class VectorDB:
    """
    ChromaDB-backed vector store for municipal regulation PDFs.

    Mirrors the interface of the Anthropic VectorDB recipe but uses:
      - ChromaDB PersistentClient instead of Voyage AI + pickle
      - OpenAI text-embedding-3-small for embeddings
      - LangChain PyPDFLoader + RecursiveCharacterTextSplitter for ingestion
    """

    # ------------------------------------------------------------------ #
    #  Construction                                                        #
    # ------------------------------------------------------------------ #

    def __init__(
        self,
        name: str,
        data_dir: Path,
        api_key: str | None = None,
        chroma_path: str = "./data/chroma_store",
        recursive: bool = False,
    ):
        """
        Args:
            name:        ChromaDB collection name — also used as a unique key on disk.
            data_dir:    Directory containing the PDF files to index.
            api_key:     OpenAI API key (falls back to OPENAI_API_KEY env var).
            chroma_path: Folder where ChromaDB stores its on-disk index.
        """
        if api_key is None:
            api_key = os.environ["OPENAI_API_KEY"]

        self.name       = name
        self.data_dir   = data_dir
        self._recursive = recursive
        self._api_key   = api_key

        # PersistentClient means the index survives kernel restarts
        self._chroma     = chromadb.PersistentClient(path=chroma_path)
        self._embed_fn   = OpenAIEmbeddingFunction(
            api_key=api_key, model_name=EMBEDDING_MODEL
        )
        self._collection = self._chroma.get_or_create_collection(
            name=name, embedding_function=self._embed_fn
        )
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=1500,
            chunk_overlap=200,
            separators=["\n\n", "\n", " ", ""],
        )
        self._enc = tiktoken.get_encoding("cl100k_base")

        print(f"VectorDB '{name}' ready — {self._collection.count():,} chunks on disk")

    # ------------------------------------------------------------------ #
    #  Public API                                                          #
    # ------------------------------------------------------------------ #

    # This function is used to discover the PDF paths in the data directory.  
    # It is used to determine if there are any new PDFs to index. 
    def _discover_pdf_paths(self) -> list[Path]:
        pattern = "**/*.pdf" if self._recursive else "*.pdf"
        return sorted(self.data_dir.glob(pattern))

    # This function is used to create a stable source id for metadata (includes subfolder). 
    def _source_key(self, pdf_path: Path) -> str:
        """Stable source id for metadata (includes subfolder)."""
        try:
            return str(pdf_path.relative_to(self.data_dir))
        except ValueError:
            return pdf_path.name

    # This function is used to get the set of sources that have already been indexed.  
    # It is used to determine if there are any new PDFs to index.    
    def _indexed_sources(self) -> set[str]:
        if self._collection.count() == 0:
            return set()
        result = self._collection.get(include=["metadatas"])
        return {m["source"] for m in result["metadatas"]}

    # This function is used to load the PDFs from the data directory into the collection.  
    # It is used to determine if there are any new PDFs to index. 
    def load_data(self) -> None:
        """
        Load PDFs from data_dir into the collection.

        Incremental: indexes only PDFs not yet present (by relative path).
        """
        pdf_paths = self._discover_pdf_paths()
        if not pdf_paths:
            raise FileNotFoundError(f"No PDFs found under {self.data_dir}")

        indexed = self._indexed_sources()
        indexed_names = {Path(s).name for s in indexed}
        missing = [
            p for p in pdf_paths
            if self._source_key(p) not in indexed and p.name not in indexed_names
        ]

        print(f"PDFs on disk: {len(pdf_paths)} | already indexed: {len(indexed)} | to index: {len(missing)}")
        if not missing:
            print(f"Collection '{self.name}' up to date ({self._collection.count():,} chunks).")
            return

        for p in missing:
            print(f"  + {self._source_key(p)}")

        raw_docs = []
        for pdf_path in missing:
            raw_docs.extend(PyPDFLoader(str(pdf_path)).load())
        print(f"Loaded {len(raw_docs)} pages from {len(missing)} PDFs")

        chunks = self._splitter.split_documents(raw_docs)

        next_index = self._collection.count()
        sections = []
        for doc in chunks:
            source_path = Path(doc.metadata.get("source", ""))
            try:
                source = str(source_path.relative_to(self.data_dir.resolve()))
            except ValueError:
                source = source_path.name
            sections.append(
                {
                    "text": doc.page_content,
                    "source": source,
                    "chunk_index": next_index,
                    "heading": _extract_heading(
                        doc.page_content, doc.metadata.get("page", 0)
                    ),
                }
            )
            next_index += 1

        print(
            f"Split into {len(sections):,} new chunks "
            f"(avg {sum(len(s['text']) for s in sections) // len(sections)} chars/chunk)"
        )
        self._embed_and_store(sections)

    # This function is used to search the collection for the top-n chunks that are most similar to the query.     
    def search(self, query: str, n_results: int = 7) -> list[dict]:
        """
        Embed query with OpenAI and return top-n chunks.

        Returns:
            List of dicts with keys: text, metadata, score (0–1, higher = more similar).
        """
        results = self._collection.query(
            query_texts=[query],
            n_results=n_results,
            include=["documents", "metadatas", "distances"],
        )
        return [
            {
                "text":     text,
                "metadata": meta,
                "score":    round(1 - dist, 4),
            }
            for text, meta, dist in zip(
                results["documents"][0],
                results["metadatas"][0],
                results["distances"][0],
            )
        ]

    # ------------------------------------------------------------------ #
    #  Private helpers                                                     #
    # ------------------------------------------------------------------ #

    # This function is used to embed and store the chunks in the collection.  
    def _embed_and_store(
        self,
        sections: list[dict],
        batch_size: int = 500,
        max_tokens: int = 7_500,
    ) -> None:
        """
        Batch-embed texts with the OpenAI API and upsert them into ChromaDB.

        Args:
            sections:   List of dicts (text, source, chunk_index, heading).
            batch_size: Max documents per OpenAI embeddings request.
            max_tokens: Token ceiling per document (OpenAI limit is 8,191).
        """
        oa = openai_client.OpenAI(api_key=self._api_key)

        # Sanitize and truncate every chunk up front
        texts = [self._sanitize(s["text"], max_tokens) for s in sections]

        # Embed in controlled batches (OpenAI caps at 2,048 inputs per request)
        embeddings: list[list[float]] = []
        for i in range(0, len(texts), batch_size):
            batch    = texts[i : i + batch_size]
            response = oa.embeddings.create(input=batch, model=EMBEDDING_MODEL)
            embeddings.extend([r.embedding for r in response.data])
            print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks")

        # Upsert into our vector database, ChromaDB
        self._collection.add(
            documents=texts,
            embeddings=embeddings,
            metadatas=[
                {
                    "source":      s["source"],
                    "chunk_index": s["chunk_index"],
                    "heading":     s["heading"],
                }
                for s in sections
            ],
            ids=[f"{s['source'].replace('/', '__')}_chunk_{s['chunk_index']}" for s in sections],
        )
        print(f"Stored {len(sections):,} chunks in collection '{self.name}'")


    def _sanitize(self, text: str, max_tokens: int | None = None) -> str:
        """Strip null bytes and truncate to max_tokens if provided."""
        text = text.replace("\x00", "").encode("utf-8", errors="ignore").decode("utf-8").strip()
        if max_tokens:
            tokens = self._enc.encode(text)
            if len(tokens) > max_tokens:
                text = self._enc.decode(tokens[:max_tokens])
        return text

DOCUMENT_CONTEXT_PROMPT = """
<document>
{doc_content}
</document>
"""

CHUNK_CONTEXT_PROMPT = """
Here is the chunk we want to situate within the whole document
<chunk>
{chunk_content}
</chunk>

Please give a short succinct context to situate this chunk within the overall document for the purposes of improving search retrieval of the chunk.
Answer only with the succinct context and nothing else.
"""

# This class is used to contextualize the chunks before indexing.  
class ContextualVectorDB(VectorDB):
    """
    VectorDB with Anthropic contextual embeddings before indexing.

    For each chunk, Claude generates a short situating description using the
    full source PDF. The full PDF is sent with prompt caching so later chunks
    from the same file read from cache instead of re-billing the full document.
    """

    def __init__(
        self,
        name: str,
        data_dir: Path,
        api_key: str | None = None,
        anthropic_api_key: str | None = None,
        chroma_path: str = "./data/chroma_store",
        contextualize_model: str | None = None,
        recursive: bool = False,
    ):
        super().__init__(name=name, data_dir=data_dir, api_key=api_key, chroma_path=chroma_path, recursive=recursive)
        if anthropic_api_key is None:
            anthropic_api_key = os.environ["ANTHROPIC_API_KEY"]
        self._anthropic = anthropic.Anthropic(api_key=anthropic_api_key)
        self._contextualize_model = contextualize_model or CONTEXTUALIZE_MODEL
        self._token_counts = {
            "input": 0,
            "output": 0,
            "cache_read": 0,
            "cache_creation": 0,
        }

    def _is_contextual_index(self) -> bool:
        """True if collection is empty or was built with contextual embeddings."""
        if self._collection.count() == 0:
            return True
        sample = self._collection.get(limit=1, include=["metadatas"])
        return bool(sample["metadatas"][0].get("contextualized_content"))

    # This function is used to reset the collection for contextual re-ingest.  
    def _reset_collection(self) -> None:
        print(f"Resetting collection '{self.name}' for contextual re-ingest...")
        self._chroma.delete_collection(self.name)
        self._collection = self._chroma.get_or_create_collection(
            name=self.name, embedding_function=self._embed_fn
        )

    def load_data(self) -> None:
        """Contextualize and embed PDFs not yet in the collection (incremental)."""
        if not self._is_contextual_index():
            self._reset_collection()

        pdf_paths = self._discover_pdf_paths()
        if not pdf_paths:
            raise FileNotFoundError(f"No PDFs found under {self.data_dir}")

        indexed = self._indexed_sources()
        indexed_names = {Path(s).name for s in indexed}
        missing = [
            p for p in pdf_paths
            if self._source_key(p) not in indexed and p.name not in indexed_names
        ]

        print(f"PDFs on disk: {len(pdf_paths)} | already indexed: {len(indexed)} | to index: {len(missing)}")
        if not missing:
            print(f"Collection '{self.name}' up to date ({self._collection.count():,} chunks).")
            return

        for p in missing:
            print(f"  + {self._source_key(p)}")

        total_new_chunks = 0
        global_chunk_index = self._collection.count()

        for pdf_idx, pdf_path in enumerate(missing, start=1):
            source = self._source_key(pdf_path)
            pages = PyPDFLoader(str(pdf_path)).load()
            doc_content = self._sanitize("\n\n".join(p.page_content for p in pages))
            doc_content = self._sanitize(doc_content, MAX_DOC_TOKENS)

            chunks = self._splitter.split_documents(pages)
            print(
                f"\n[{pdf_idx}/{len(missing)}] Contextualizing {len(chunks)} chunks from {source}...",
                flush=True,
            )

            doc_cache_read_before = self._token_counts["cache_read"]
            sections: list[dict] = []

            for chunk_idx, chunk in enumerate(chunks, start=1):
                context = self._situate_context(doc_content, chunk.page_content)
                original = chunk.page_content
                sections.append(
                    {
                        "text": original,
                        "text_to_embed": f"{context}\n\n{original}",
                        "contextualized_content": context,
                        "source": source,
                        "chunk_index": global_chunk_index,
                        "heading": _extract_heading(
                            original, chunk.metadata.get("page", 0)
                        ),
                    }
                )
                global_chunk_index += 1

                if chunk_idx == 1 or chunk_idx % 25 == 0 or chunk_idx == len(chunks):
                    print(
                        f"    chunk {chunk_idx}/{len(chunks)} contextualized",
                        flush=True,
                    )

            doc_cache_read = self._token_counts["cache_read"] - doc_cache_read_before
            print(f"  Embedding + storing {len(sections)} chunks for {source}...", flush=True)
            self._embed_and_store(sections)
            total_new_chunks += len(sections)
            print(
                f"  Done {source}: stored {len(sections)} chunks | "
                f"cache read {doc_cache_read:,} token reads | "
                f"collection total {self._collection.count():,}",
                flush=True,
            )

        self._print_contextualization_stats(total_new_chunks)

    def search(self, query: str, n_results: int = 7) -> list[dict]:
        """Return original chunk text for display even though embeddings used context."""
        hits = super().search(query, n_results=n_results)
        for hit in hits:
            original = hit["metadata"].get("original_content")
            if original:
                hit["text"] = original
        return hits
# This function is used to situate the context of the chunk within the overall document.  
# It is used to improve retrieval rates by adding additional document "context" to embeddings. 
    def _situate_context(self, doc_content: str, chunk_content: str) -> str:
        response = self._anthropic.messages.create(
            model=self._contextualize_model,
            max_tokens=300,
            temperature=0.0,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": DOCUMENT_CONTEXT_PROMPT.format(doc_content=doc_content),
                            "cache_control": {"type": "ephemeral"},
                        },
                        {
                            "type": "text",
                            "text": CHUNK_CONTEXT_PROMPT.format(chunk_content=chunk_content),
                        },
                    ],
                }
            ],
        )
        usage = response.usage
        self._token_counts["input"] += usage.input_tokens
        self._token_counts["output"] += usage.output_tokens
        self._token_counts["cache_read"] += getattr(usage, "cache_read_input_tokens", 0) or 0
        self._token_counts["cache_creation"] += getattr(usage, "cache_creation_input_tokens", 0) or 0
        return response.content[0].text.strip()

    def _print_contextualization_stats(self, total_chunks: int) -> None:
        tc = self._token_counts
        total_input = tc["input"] + tc["cache_read"] + tc["cache_creation"]
        savings = (tc["cache_read"] / total_input * 100) if total_input else 0
        print(f"\nContextualized {total_chunks:,} chunks")
        print(f"  Input tokens (non-cache): {tc['input']:,}")
        print(f"  Output tokens: {tc['output']:,}")
        print(f"  Cache creation tokens: {tc['cache_creation']:,}")
        print(f"  Cache read tokens: {tc['cache_read']:,}")
        print(f"  Cache read share: {savings:.1f}% of all input tokens")

    def _embed_and_store(
        self,
        sections: list[dict],
        batch_size: int = 500,
        max_tokens: int = 7_500,
    ) -> None:
        """Embed contextualized text; store original text in metadata for citations."""
        oa = openai_client.OpenAI(api_key=self._api_key)

        texts = [
            self._sanitize(s.get("text_to_embed", s["text"]), max_tokens)
            for s in sections
        ]

        embeddings: list[list[float]] = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            response = oa.embeddings.create(input=batch, model=EMBEDDING_MODEL)
            embeddings.extend([r.embedding for r in response.data])
            print(f"  Embedded {min(i + batch_size, len(texts))}/{len(texts)} chunks")

        self._collection.add(
            documents=texts,
            embeddings=embeddings,
            metadatas=[
                {
                    "source": s["source"],
                    "chunk_index": s["chunk_index"],
                    "heading": s["heading"],
                    "original_content": s["text"],
                    "contextualized_content": s.get("contextualized_content", ""),
                }
                for s in sections
            ],
            ids=[f"{s['source'].replace('/', '__')}_chunk_{s['chunk_index']}" for s in sections],
        )
        print(f"Stored {len(sections):,} contextualized chunks in '{self.name}'")


In [ ]:
# Contextual ingest: Claude situates each chunk (prompt caching), then OpenAI embeds.
# Saves after each PDF. Expect ~30-90 min for all 32 PDFs (~2,823 chunks).
# Baseline (fast) index remains in regulations_openai_baseline.
db = ContextualVectorDB(
    name="regulations_openai_contextual",
    data_dir=Path("data"),
    recursive=True,
)
db.load_data()

## Interactive RAG Chat

Uses contextual embeddings for retrieval (`db.search()`), but displays **original**
chunk text in answers and citations.

**Session commands:**
| Command | Effect |
|---------|--------|
| `quit` / `q` | End the session |
| `clear` | Wipe conversation memory and reset token counter |

In [4]:
# Governance config (override via .env)
TOKEN_BUDGET = int(os.environ.get("RAG_TOKEN_BUDGET", 50_000))
TIMEOUT_SECS = int(os.environ.get("RAG_TIMEOUT_SECS", 30))
LOG_FILE     = Path(os.environ.get("RAG_LOG_FILE", "rag_session_log.csv"))
COLLECTION_NAME = "regulations_openai_contextual"
_rag_db: ContextualVectorDB | None = None


def _get_db() -> ContextualVectorDB:
    """Open persisted Chroma collection (no re-ingest)."""
    global _rag_db
    if _rag_db is None:
        _rag_db = ContextualVectorDB(
            name=COLLECTION_NAME,
            data_dir=Path("data"),
            recursive=True,
        )
    return _rag_db

# Session state (persists across re-runs of this cell)
conversation_history: list[dict] = []
session_tokens_used: int = 0


def _log_turn(query: str, hits: list[dict], answer: str, tokens: int) -> None:
    """Append one row per query to the CSV log file."""
    is_new = not LOG_FILE.exists()
    n = len(hits)
    chunk_headers = [
        f"chunk_{i+1}_{f}" for i in range(n) for f in ("score", "source", "heading", "text")
    ]
    fieldnames = ["timestamp", "query", "answer", "tokens_turn"] + chunk_headers

    row = {
        "timestamp":   datetime.now(timezone.utc).isoformat(),
        "query":       query,
        "answer":      answer,
        "tokens_turn": tokens,
    }
    for i, hit in enumerate(hits):
        p = f"chunk_{i+1}_"
        row[p + "score"]   = hit["score"]
        row[p + "source"]  = hit["metadata"]["source"]
        row[p + "heading"] = hit["metadata"]["heading"]
        row[p + "text"]    = hit["text"].replace("\n", " ")

    with open(LOG_FILE, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if is_new:
            writer.writeheader()
        writer.writerow(row)
    print(f"[Logged to {LOG_FILE}]")


def compare_interactive(n_results: int = 7) -> None:
    """
    Interactive RAG chat session.

    Retrieval is handled by `db.search()`; answers come from Claude.
    Session memory grows turn-by-turn and is sent as prior context.
    """
    global conversation_history, session_tokens_used

    rag_db = _get_db()
    ac = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    print(f"\nRAG Chat  |  collection: '{rag_db.name}'  |  model: {LLM_MODEL}")
    print(f"Governance: token budget={TOKEN_BUDGET:,}  |  timeout={TIMEOUT_SECS}s")
    print("Commands: 'quit' to exit  |  'clear' to reset memory\n")

    while True:
        remaining = TOKEN_BUDGET - session_tokens_used
        if remaining <= 0:
            print(
                f"[Session token budget of {TOKEN_BUDGET:,} exhausted. "
                "Type 'clear' to reset or 'quit' to exit.]"
            )

        query = input("You: ").strip()
        if not query:
            continue
        if query.lower() in ("quit", "exit", "q"):
            print(f"Session ended. Tokens used: {session_tokens_used:,}")
            break
        if query.lower() == "clear":
            conversation_history.clear()
            session_tokens_used = 0
            print("[Memory cleared. Token counter reset.]\n")
            continue
        if session_tokens_used >= TOKEN_BUDGET:
            print("[Cannot process — token budget exhausted. Type 'clear' to reset.]\n")
            continue

        # Retrieve relevant chunks via the VectorDB class
        hits = rag_db.search(query, n_results=n_results)

        context_str = "\n\n".join(
            f"[{i+1}] {h['metadata']['source']} | {h['metadata']['heading']}\n{h['text']}"
            for i, h in enumerate(hits)
        )

        # Build message list from growing conversation history
        messages = []
        for turn in conversation_history:
            messages.append({"role": "user",      "content": turn["question"]})
            messages.append({"role": "assistant",  "content": turn["answer"]})
        messages.append({"role": "user", "content": query})

        system_prompt = (
            "You are a precise assistant answering questions about Baldwin Borough and "
            "Allegheny County municipal regulations. You must follow these rules strictly:\n\n"
            "1. Answer ONLY from the numbered context passages below. "
            "Do NOT use your training data or general knowledge.\n"
            "2. Every factual claim MUST be followed immediately by an inline "
            "citation in the form [N] referencing the passage number it came from.\n"
            "3. If a question cannot be answered from the context, say exactly: "
            "'This information is not available in the provided documents.'\n"
            "4. Never speculate or infer beyond what the passages explicitly state.\n\n"
            f"CONTEXT PASSAGES:\n{context_str}"
        )

        # Call Claude with a hard timeout
        try:
            with ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(
                    ac.messages.create,
                    model=LLM_MODEL,
                    max_tokens=1024,
                    system=system_prompt,
                    messages=messages,
                )
                response = future.result(timeout=TIMEOUT_SECS)
        except FuturesTimeoutError:
            print(f"\n[Request timed out after {TIMEOUT_SECS}s. Please try again.]\n")
            continue
        except Exception as e:
            print(f"\n[API error: {e}]\n")
            continue

        turn_tokens      = response.usage.input_tokens + response.usage.output_tokens
        session_tokens_used += turn_tokens
        remaining_after  = TOKEN_BUDGET - session_tokens_used
        answer           = response.content[0].text

        print(f"\n{'─' * 60}")
        print(f"You asked: {query}")
        print(f"{'─' * 60}")
        print(f"\nAssistant: {answer}")

        print("\n" + "─" * 60)
        print("Retrieved passages (cited as [N] above):")
        for i, hit in enumerate(hits):
            print(
                f"\n  [{i+1}] {hit['metadata']['source']} | "
                f"{hit['metadata']['heading']}  (score: {hit['score']})"
            )
            print(f"  {hit['text']}")
        print("\n" + "─" * 60)
        print(
            f"[Tokens this turn: {turn_tokens:,} | "
            f"Session total: {session_tokens_used:,} | "
            f"Budget remaining: {remaining_after:,}]\n"
        )

        conversation_history.append({"question": query, "answer": answer})
        _log_turn(query, hits, answer, turn_tokens)


compare_interactive()

VectorDB 'regulations_openai_contextual' ready — 2,823 chunks on disk

RAG Chat  |  collection: 'regulations_openai_contextual'  |  model: claude-sonnet-4-5
Governance: token budget=50,000  |  timeout=20s
Commands: 'quit' to exit  |  'clear' to reset memory


────────────────────────────────────────────────────────────
You asked: can i have a firepit in my backyard in baldwin
────────────────────────────────────────────────────────────

Assistant: Based on the provided documents, you can have a firepit in your backyard in Baldwin, subject to these restrictions:

**Requirements for firepits:**

1. **Only clean, dry wood products may be burned** [1]

2. **Burning must be in an approved container**: The fire must be in "an approved outdoor rated fireplace, fire pit, chimney or fifty-five-gallon drum" [1]

3. **Distance from property line**: "All burning and fires must be no less than 15 feet distant from the nearest property line" [1]

4. **Supervision required**: "All outdoor burning sha